# Customer Churn Prediction - Model Training & Evaluation

This notebook trains and evaluates multiple machine learning models to predict customer churn.

**Objectives:**
- Prepare data for modeling
- Engineer relevant features
- Train multiple models
- Evaluate and compare model performance
- Select and save the best model
- Generate insights

## 1. Import Libraries and Load Data

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_telco_data, clean_data, validate_data
from src.feature_engineering import (
    separate_features_and_target, encode_categorical_features, 
    engineer_features, scale_features, get_feature_importance_dataframe
)
from src.model_utils import (
    train_logistic_regression, train_random_forest, evaluate_model, 
    compare_models, select_best_model, save_model, cross_validate_model
)

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 2. Load and Prepare Data

In [ ]:
# Load dataset
df = load_telco_data('../data/WA_Fn-UseC_-_Telco_Customer_Churn.csv')

# Clean data
df = clean_data(df)

# Validate data
validate_data(df)

print(f"\nDataset shape: {df.shape}")
print(f"Churn distribution:\n{df['Churn'].value_counts()}")

## 3. Feature Engineering

In [ ]:
# Separate features and target
X, y = separate_features_and_target(df)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts()}")
print(f"\nFeatures:\n{X.dtypes}")

In [ ]:
# Engineer features
X, _ = engineer_features(X, None)

print(f"\n✓ Features engineered")
print(f"New shape: {X.shape}")
print(f"\nNew features: {X.columns.tolist()}")

## 4. Train-Test Split

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Testing set size: {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nTraining set churn distribution:\n{y_train.value_counts()}")
print(f"\nTesting set churn distribution:\n{y_test.value_counts()}")

## 5. Encode Categorical Features

In [ ]:
# Encode categorical features
X_train_encoded, X_test_encoded, encoders = encode_categorical_features(X_train, X_test, fit_encoders=True)

print(f"✓ Categorical features encoded")
print(f"\nTrain shape: {X_train_encoded.shape}")
print(f"Test shape: {X_test_encoded.shape}")
print(f"\nAll features are now numeric: {X_train_encoded.dtypes.unique().tolist()}")

## 6. Feature Scaling

In [ ]:
# Identify numeric columns for scaling
numeric_cols = X_train_encoded.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Scale features
X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train_encoded, X_test_encoded, numeric_cols=numeric_cols
)

print(f"✓ Features scaled using StandardScaler")
print(f"\nScaled features sample (first 5 rows):")
print(X_train_scaled.head())

## 7. Train Logistic Regression Model

In [ ]:
# Train model
lr_model = train_logistic_regression(X_train_scaled, y_train)

# Evaluate
lr_results = evaluate_model(
    lr_model, X_train_scaled, y_train, X_test_scaled, y_test, 
    model_name='Logistic Regression'
)

print("\n" + "="*60)
print("LOGISTIC REGRESSION - Training Metrics")
print("="*60)
for metric, value in lr_results['train_metrics'].items():
    print(f"{metric.upper():15s}: {value:.4f}")

print("\n" + "="*60)
print("LOGISTIC REGRESSION - Testing Metrics")
print("="*60)
for metric, value in lr_results['test_metrics'].items():
    print(f"{metric.upper():15s}: {value:.4f}")

## 8. Train Random Forest Model

## 9. Model Comparison

In [ ]:
# Prepare comparison
models_results = {
    'Logistic Regression': lr_results,
    'Random Forest': rf_results
}

# Compare models
comparison_df = compare_models(models_results)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models = comparison_df['Model'].tolist()

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    values = comparison_df[metric].tolist()
    ax.bar(models, values, color=['#3498db', '#e74c3c'], alpha=0.7)
    ax.set_title(f'{metric} Comparison', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for i, v in enumerate(values):
        ax.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../screenshots/05_model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Model comparison visualization saved")

## 10. Confusion Matrix Analysis

## 11. Feature Importance

In [ ]:
# Get feature importance from Random Forest
feature_importance = rf_model.feature_importances_
feature_names = X_train_scaled.columns.tolist()

importance_df = get_feature_importance_dataframe(feature_names, feature_importance, top_n=15)

print("\nTOP 15 MOST IMPORTANT FEATURES:")
print(importance_df.to_string(index=False))

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(range(len(importance_df)), importance_df['Importance'].values, color='#3498db', alpha=0.7)
ax.set_yticks(range(len(importance_df)))
ax.set_yticklabels(importance_df['Feature'].values)
ax.set_xlabel('Importance', fontweight='bold')
ax.set_title('Top 15 Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../screenshots/07_feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Feature importance visualization saved")

## 12. Cross-Validation

## 13. Select and Save Best Model

## 14. Model Performance Summary

## 15. Business Insights

## 16. Model Artifacts Summary

## 17. Export Results

In [ ]:
print("\n" + "="*80)
print("✓ NOTEBOOK COMPLETED SUCCESSFULLY")
print("="*80)
print("\nNext steps:")
print("1. Review the saved model and artifacts")
print("2. Run the Streamlit app to make predictions")
print("3. Monitor model performance in production")